# Track A — Rulman Bazlı Gerçek ve Tahmin Edilen RUL Analizi
---
Bu notebook, **XGBoost** ve **Random Forest** modellerinin validation ve test rulmanları üzerindeki anlık tahminlerini (RUL) pencere bazlı olarak yan yana karşılaştırmalı tablolar halinde gösterir.

Böylece modellerin degradasyonu ne aşamada yakaladığını, her bir rulman için sayısal olarak karşılaştırabiliriz.

In [1]:
import sys
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# Proje kök dizinini ekle
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import ACTUAL_RUL_SECONDS
from evaluation import evaluate_test_bearings

## 1. Veri ve Modellerin Yüklenmesi

In [2]:
MODELS_DIR = PROJECT_ROOT / "experiments" / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Modelleri yükle
prep = joblib.load(MODELS_DIR / "preprocessor.pkl")
rf_model = joblib.load(MODELS_DIR / "Random_Forest.pkl")
xgb_model = joblib.load(MODELS_DIR / "XGBoost.pkl")

# Özellik dosyalarını yükle
df_train_full = pd.read_csv(PROCESSED_DIR / "features_train.csv")
df_test_full = pd.read_csv(PROCESSED_DIR / "features_test.csv")

print("Modeller, preprocessor ve özellik verileri başarıyla yüklendi!")

Modeller, preprocessor ve özellik verileri başarıyla yüklendi!


## 2. RUL Standartlaştırma Fonksiyonu
Gerçek RUL değerlerini 125.0 dakikaya sabitleyerek (piecewise linear RUL) standartlaştırıyoruz.

In [3]:
def apply_standard_rul(df, max_rul=125.0):
    total_life = df.groupby('bearing').apply(lambda g: (g['time_s'] + g['rul_s']).max(), include_groups=False)
    total_map = df['bearing'].map(total_life)
    df['rul_min'] = np.clip((total_map - df['time_s']) / 60.0, 0, max_rul)
    return df

df_train_full = apply_standard_rul(df_train_full)
df_test_full = apply_standard_rul(df_test_full)
print("RUL sütunları başarıyla oluşturuldu!")

RUL sütunları başarıyla oluşturuldu!


## 3. Doğrulama (Validation) Rulmanları Analizi
Doğrulama rulmanları (`Bearing1_2`, `Bearing2_2`, `Bearing3_2`) tam ömür verileridir.
Her bir rulman için:
1. **Tüm Seyir Özeti (Her 30. pencere)**: Başlangıçtaki sağlıklı fazdan arızaya gidiş seyrini görmek için.
2. **Arıza Öncesi Son 15 Pencere**: Modellerin arıza anına yaklaşırken tahmin doğruluğunu görmek için.
Tablolarda XGBoost ve Random Forest tahminleri gerçek RUL ile yan yana listelenmektedir.

In [4]:
val_bearings = ["Bearing1_2", "Bearing2_2", "Bearing3_2"]

for bname in val_bearings:
    bdf = df_train_full[df_train_full["bearing"] == bname].sort_values("time_s").copy()
    
    # Preprocessing ve Predict
    X_b, _ = prep.transform(bdf)
    pred_xgb = xgb_model.predict(X_b)
    pred_rf = rf_model.predict(X_b)
    
    # Sonuç DataFrame'i oluştur
    res_df = pd.DataFrame({
        "Pencere No": np.arange(len(bdf)),
        "Zaman (Dk)": np.round(bdf["time_s"] / 60.0, 2),
        "Gerçek RUL (Dk)": np.round(bdf["rul_min"], 2),
        "XGBoost Tahmin (Dk)": np.round(pred_xgb, 2),
        "XGBoost Hata (Dk)": np.round(pred_xgb - bdf["rul_min"].values, 2),
        "RF Tahmin (Dk)": np.round(pred_rf, 2),
        "RF Hata (Dk)": np.round(pred_rf - bdf["rul_min"].values, 2)
    })
    
    print("\n" + "="*90)
    print(f"  RULMAN: {bname} (Toplam {len(bdf)} pencere)")
    print("="*90)
    
    print(f"\n---> {bname} - Tüm Seyir Seyri (Her 30. Pencere):")
    display(res_df.iloc[::30])
    
    print(f"\n---> {bname} - Arıza Öncesi Son 15 Pencere:")
    display(res_df.tail(15))
    print("="*90)


  RULMAN: Bearing1_2 (Toplam 871 pencere)

---> Bearing1_2 - Tüm Seyir Seyri (Her 30. Pencere):


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
2803,0,0.0,125.00,84.230003,-40.77,80.12,-44.88
2833,30,5.0,125.00,77.029999,-47.97,74.67,-50.33
2863,60,10.0,125.00,73.059998,-51.94,70.50,-54.50
2893,90,15.0,125.00,69.400002,-55.60,62.35,-62.65
2923,120,20.0,125.00,49.630001,-75.37,53.19,-71.81
2953,150,25.0,120.17,44.090000,-76.08,49.83,-70.34
2983,180,30.0,115.17,38.680000,-76.49,37.38,-77.78
3013,210,35.0,110.17,36.299999,-73.87,36.49,-73.67
3043,240,40.0,105.17,36.990002,-68.17,48.80,-56.37
3073,270,45.0,100.17,28.480000,-71.69,30.02,-70.14



---> Bearing1_2 - Arıza Öncesi Son 15 Pencere:


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
3659,856,142.67,2.50,6.39,3.89,4.03,1.53
3660,857,142.83,2.33,6.41,4.07,4.14,1.80
3661,858,143.00,2.17,6.18,4.01,4.36,2.19
3662,859,143.17,2.00,6.25,4.25,4.34,2.34
3663,860,143.33,1.83,6.27,4.44,4.56,2.73
3664,861,143.50,1.67,6.20,4.53,4.52,2.85
3665,862,143.67,1.50,7.59,6.09,6.67,5.17
3666,863,143.83,1.33,6.90,5.56,6.77,5.44
3667,864,144.00,1.17,5.03,3.86,4.93,3.77
3668,865,144.17,1.00,5.20,4.20,6.58,5.58



  RULMAN: Bearing2_2 (Toplam 797 pencere)

---> Bearing2_2 - Tüm Seyir Seyri (Her 30. Pencere):


...

,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
5367,782,130.33,2.50,5.83,3.33,4.61,2.11
5368,783,130.50,2.33,4.96,2.63,4.29,1.96
5369,784,130.67,2.17,5.18,3.01,4.35,2.18
5370,785,130.83,2.00,5.23,3.23,4.02,2.02
5371,786,131.00,1.83,5.09,3.25,3.82,1.99
5372,787,131.17,1.67,4.45,2.79,3.77,2.10
5373,788,131.33,1.50,4.30,2.80,3.77,2.27
5374,789,131.50,1.33,4.52,3.19,3.72,2.39
5375,790,131.67,1.17,3.02,1.85,2.13,0.96
5376,791,131.83,1.00,3.01,2.01,2.33,1.33



  RULMAN: Bearing3_2 (Toplam 1637 pencere)

---> Bearing3_2 - Tüm Seyir Seyri (Her 30. Pencere):


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
5897,0,0.0,125.00,83.690002,-41.31,86.22,-38.78
5927,30,5.0,125.00,85.430000,-39.57,85.65,-39.35
5957,60,10.0,125.00,85.150002,-39.85,83.20,-41.80
5987,90,15.0,125.00,84.160004,-40.84,84.17,-40.83
6017,120,20.0,125.00,59.020000,-65.98,53.44,-71.56
6047,150,25.0,125.00,44.459999,-80.54,49.23,-75.77
6077,180,30.0,125.00,43.200001,-81.80,48.36,-76.64
6107,210,35.0,125.00,36.029999,-88.97,35.58,-89.42
6137,240,40.0,125.00,37.730000,-87.27,40.86,-84.14
6167,270,45.0,125.00,35.779999,-89.22,36.56,-88.44



---> Bearing3_2 - Arıza Öncesi Son 15 Pencere:


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
7519,1622,270.33,2.50,5.34,2.84,6.24,3.74
7520,1623,270.50,2.33,5.95,3.62,8.07,5.74
7521,1624,270.67,2.17,5.88,3.72,7.52,5.35
7522,1625,270.83,2.00,5.90,3.90,7.82,5.82
7523,1626,271.00,1.83,5.89,4.06,8.32,6.48
7524,1627,271.17,1.67,5.58,3.92,7.69,6.03
7525,1628,271.33,1.50,5.54,4.04,7.22,5.72
7526,1629,271.50,1.33,5.48,4.15,7.12,5.79
7527,1630,271.67,1.17,5.44,4.27,7.25,6.08
7528,1631,271.83,1.00,5.52,4.52,7.30,6.30


## 4. Test Rulmanları Analizi
Test rulmanları belirli bir arıza aşamasına kadar çalıştırılıp kesilmiştir.
Her bir test rulmanı için:
1. **Tüm Seyir Özeti (Her 30. pencere)**.
2. **Kesilme Öncesi Son 10 Pencere**.
Tahminler yan yana listelenmektedir.

In [5]:
test_bearings = sorted(df_test_full["bearing"].unique())

for bname in test_bearings:
    bdf = df_test_full[df_test_full["bearing"] == bname].sort_values("time_s").copy()
    
    # Preprocessing ve Predict
    X_b, _ = prep.transform(bdf)
    pred_xgb = xgb_model.predict(X_b)
    pred_rf = rf_model.predict(X_b)
    
    # Sonuç DataFrame'i oluştur
    res_df = pd.DataFrame({
        "Pencere No": np.arange(len(bdf)),
        "Zaman (Dk)": np.round(bdf["time_s"] / 60.0, 2),
        "Gerçek RUL (Dk)": np.round(bdf["rul_min"], 2),
        "XGBoost Tahmin (Dk)": np.round(pred_xgb, 2),
        "XGBoost Hata (Dk)": np.round(pred_xgb - bdf["rul_min"].values, 2),
        "RF Tahmin (Dk)": np.round(pred_rf, 2),
        "RF Hata (Dk)": np.round(pred_rf - bdf["rul_min"].values, 2)
    })
    
    print("\n" + "="*90)
    print(f"  RULMAN: {bname} (Toplam {len(bdf)} pencere)")
    print("="*90)
    
    print(f"\n---> {bname} - Tüm Seyir Seyri (Her 30. Pencere):")
    display(res_df.iloc[::30])
    
    print(f"\n---> {bname} - Kesilme Öncesi Son 10 Pencere:")
    display(res_df.tail(10))
    print("="*90)


  RULMAN: Bearing1_3 (Toplam 1802 pencere)

---> Bearing1_3 - Tüm Seyir Seyri (Her 30. Pencere):


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
0,0,0.0,125.00,89.339996,-35.66,89.93,-35.07
30,30,5.0,125.00,105.139999,-19.86,97.47,-27.53
60,60,10.0,125.00,50.849998,-74.15,48.88,-76.12
90,90,15.0,125.00,56.209999,-68.79,62.15,-62.85
120,120,20.0,125.00,68.080002,-56.92,54.08,-70.92
...,...,...,...,...,...,...,...
1680,1680,280.0,115.83,90.970001,-24.86,85.54,-30.29
1710,1710,285.0,110.83,103.010002,-7.82,105.45,-5.39
1740,1740,290.0,105.83,102.650002,-3.18,108.31,2.48
1770,1770,295.0,100.83,99.330002,-1.51,104.35,3.52



---> Bearing1_3 - Kesilme Öncesi Son 10 Pencere:


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
1792,1792,298.67,97.17,97.379997,0.22,103.44,6.28
1793,1793,298.83,97.00,97.519997,0.52,103.40,6.40
1794,1794,299.00,96.83,97.169998,0.34,103.46,6.63
1795,1795,299.17,96.67,97.150002,0.48,103.55,6.89
1796,1796,299.33,96.50,97.040001,0.54,103.53,7.03
1797,1797,299.50,96.33,97.320000,0.99,103.61,7.28
1798,1798,299.67,96.17,97.269997,1.10,104.49,8.32
1799,1799,299.83,96.00,97.120003,1.12,104.76,8.76
1800,1800,300.00,95.83,97.309998,1.47,104.65,8.82
1801,1801,300.17,95.67,97.400002,1.74,104.67,9.00



  RULMAN: Bearing1_4 (Toplam 1139 pencere)

---> Bearing1_4 - Tüm Seyir Seyri (Her 30. Pencere):


...

,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
13597,162,27.00,11.33,19.690001,8.36,16.86,5.52
13598,163,27.17,11.17,25.930000,14.77,28.07,16.90
13599,164,27.33,11.00,17.660000,6.66,15.35,4.35
13600,165,27.50,10.83,17.940001,7.11,14.41,3.58
13601,166,27.67,10.67,26.430000,15.76,20.70,10.03
13602,167,27.83,10.50,45.799999,35.30,36.30,25.80
13603,168,28.00,10.33,46.490002,36.16,35.12,24.78
13604,169,28.17,10.17,45.630001,35.47,33.07,22.91
13605,170,28.33,10.00,45.450001,35.45,33.15,23.15
13606,171,28.50,9.83,45.270000,35.43,33.04,23.21



  RULMAN: Bearing3_3 (Toplam 352 pencere)

---> Bearing3_3 - Tüm Seyir Seyri (Her 30. Pencere):


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
13607,0,0.0,72.33,82.919998,10.58,84.32,11.98
13637,30,5.0,67.33,86.389999,19.06,86.92,19.58
13667,60,10.0,62.33,64.220001,1.88,53.92,-8.41
13697,90,15.0,57.33,66.860001,9.52,55.51,-1.82
13727,120,20.0,52.33,66.360001,14.02,60.67,8.34
13757,150,25.0,47.33,88.309998,40.98,100.14,52.80
13787,180,30.0,42.33,65.519997,23.18,60.54,18.21
13817,210,35.0,37.33,64.879997,27.54,54.88,17.55
13847,240,40.0,32.33,69.260002,36.92,62.65,30.32
13877,270,45.0,27.33,68.059998,40.73,54.88,27.55



---> Bearing3_3 - Kesilme Öncesi Son 10 Pencere:


,Pencere No,Zaman (Dk),Gerçek RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
13949,342,57.00,15.33,58.080002,42.75,43.14,27.81
13950,343,57.17,15.17,57.759998,42.59,42.95,27.78
13951,344,57.33,15.00,81.010002,66.01,85.07,70.07
13952,345,57.50,14.83,80.139999,65.31,85.15,70.31
13953,346,57.67,14.67,80.889999,66.22,85.23,70.56
13954,347,57.83,14.50,80.019997,65.52,84.77,70.27
13955,348,58.00,14.33,80.610001,66.27,84.74,70.41
13956,349,58.17,14.17,80.430000,66.26,84.61,70.44
13957,350,58.33,14.00,80.830002,66.83,84.26,70.26
13958,351,58.50,13.83,80.290001,66.46,83.89,70.06


## 5. Test Rulmanları Final Tahmin Karşılaştırması ve PHM Skoru
Yarışma değerlendirmesinde kullanılan **en son penceredeki** tahminlerin ve gerçek RUL değerlerinin toplu karşılaştırması aşağıda tablo halinde verilmiştir.

In [6]:
summary_data = []

for bname in test_bearings:
    bdf = df_test_full[df_test_full["bearing"] == bname].sort_values("time_s").copy()
    
    # Preprocess
    X_b, _ = prep.transform(bdf)
    
    # Son pencere tahminleri
    pred_xgb = xgb_model.predict(X_b[-1:])[0]
    pred_rf = rf_model.predict(X_b[-1:])[0]
    
    actual_rul_sec = ACTUAL_RUL_SECONDS[bname]
    actual_rul_min = actual_rul_sec / 60.0
    
    summary_data.append({
        "Rulman": bname,
        "Gerçek Son RUL (Dk)": round(actual_rul_min, 2),
        "XGBoost Tahmin (Dk)": round(pred_xgb, 2),
        "XGBoost Hata (Dk)": round(pred_xgb - actual_rul_min, 2),
        "RF Tahmin (Dk)": round(pred_rf, 2),
        "RF Hata (Dk)": round(pred_rf - actual_rul_min, 2),
    })

df_summary = pd.DataFrame(summary_data)
print("\n=== KESİLME NOKTASINDAKİ (FİNAL) TAHMİNLER VE HATALAR ===")
display(df_summary)

# PHM Skorlarını hesapla
xgb_preds = {row["Rulman"]: row["XGBoost Tahmin (Dk)"] for _, row in df_summary.iterrows()}
rf_preds = {row["Rulman"]: row["RF Tahmin (Dk)"] for _, row in df_summary.iterrows()}

print("\n=== XGBOOST MODELLERİNİN PHM SKORU ===")
evaluate_test_bearings(xgb_preds, unit="min")

print("\n=== RANDOM FOREST MODELLERİNİN PHM SKORU ===")
evaluate_test_bearings(rf_preds, unit="min")


=== KESİLME NOKTASINDAKİ (FİNAL) TAHMİNLER VE HATALAR ===


,Rulman,Gerçek Son RUL (Dk),XGBoost Tahmin (Dk),XGBoost Hata (Dk),RF Tahmin (Dk),RF Hata (Dk)
0,Bearing1_3,95.50,97.400002,1.900000,104.67,9.17
1,Bearing1_4,5.65,5.010000,-0.640000,4.47,-1.18
2,Bearing1_5,26.83,56.090000,29.260000,44.76,17.93
3,Bearing1_6,24.33,35.490002,11.160000,33.49,9.16
4,Bearing1_7,126.17,55.040001,-71.129997,41.88,-84.29
5,Bearing2_3,125.50,60.820000,-64.680000,62.39,-63.11
6,Bearing2_4,23.17,22.860001,-0.300000,13.87,-9.30
7,Bearing2_5,51.50,34.650002,-16.850000,45.17,-6.33
8,Bearing2_6,21.50,54.520000,33.020000,53.02,31.52
9,Bearing2_7,9.67,45.270000,35.599998,33.04,23.38



=== XGBOOST MODELLERİNİN PHM SKORU ===

  PHM 2012 Challenge Score : 0.2748
  ----------------------------------------------------------
  Bearing          Gerçek   Tahmin     %Er      Ai
  ----------------------------------------------------------
  Bearing1_3         95.5     97.4   -2.0%  0.7590
  Bearing1_4          5.7      5.0   11.3%  0.6753
  Bearing1_5         26.8     56.1 -109.0%  0.0000
  Bearing1_6         24.3     35.5  -45.8%  0.0017
  Bearing1_7        126.2     55.0   56.4%  0.1417
  Bearing2_3        125.5     60.8   51.5%  0.1676
  Bearing2_4         23.2     22.9    1.3%  0.9552
  Bearing2_5         51.5     34.7   32.7%  0.3218
  Bearing2_6         21.5     54.5 -153.6%  0.0000
  Bearing2_7          9.7     45.3 -368.3%  0.0000
  Bearing3_3         13.7     80.3 -487.5%  0.0000

=== RANDOM FOREST MODELLERİNİN PHM SKORU ===

  PHM 2012 Challenge Score : 0.1755
  ----------------------------------------------------------
  Bearing          Gerçek   Tahmin     %Er   

{'score': 0.17548794458076344,
 'percent_errors': [np.float64(-9.602094240827643),
  np.float64(20.884955751842753),
  np.float64(-66.8074534159001),
  np.float64(-37.630136986146745),
  np.float64(66.80581241738429),
  np.float64(50.28685258960137),
  np.float64(40.12949640270448),
  np.float64(12.291262135898458),
  np.float64(-146.60465116210884),
  np.float64(-241.7931034457746),
  np.float64(-513.8292682889232)],
 'accuracies': [0.26417780200150803,
  0.48489766525837036,
  9.501463789536535e-05,
  0.005425502966108557,
  0.09873527244643114,
  0.17502796731728182,
  0.2488805131790188,
  0.6531276510906339,
  1.4911471378958137e-09,
  2.770796456949278e-15,
  1.1598331540827158e-31],
 'names': ['Bearing1_3',
  'Bearing1_4',
  'Bearing1_5',
  'Bearing1_6',
  'Bearing1_7',
  'Bearing2_3',
  'Bearing2_4',
  'Bearing2_5',
  'Bearing2_6',
  'Bearing2_7',
  'Bearing3_3'],
 'act_ruls': [95.5,
  5.65,
  26.833333333333332,
  24.333333333333332,
  126.16666666666667,
  125.5,
  23.1666666